# Data Preparation and Cleaning

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Data Understanding / Data Preparation

## Purpose

This notebook prepares the Spinner/Siemens CNC machine dataset for later analysis and machine-learning experiments. It:

1. Loads the source CSV file.
2. removes label columns that are not used in this project;
3. converts boolean columns and removes unsupported non-numeric fields;
4. handles blank and missing values;
5. exports a cleaned dataset and a min-max-scaled version;
6. explores correlations between numeric features and the target label.

The target variable is `Label_03`, representing the classes used in the project (Production vs. Changeover).

> **Data and privacy:** The source dataset is not included in this repository. Only run this notebook if you have permission to use the dataset. Do not upload confidential or proprietary machine data.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt`. Place the authorized source file at `data.csv` in the same folder as this notebook, or change `DATA_PATH` below.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

DATA_PATH = Path("../data.csv")  # Adjust if your file is stored elsewhere
CLEANED_OUTPUT_PATH = Path("../cleaned_data.csv")
SCALED_OUTPUT_PATH = Path("../normalized_data.csv")

TARGET_COLUMN = "Label_03"
TIME_COLUMN = "time"

## 2. Load and inspect the source data

The first inspection checks the dataset dimensions, column names, data types, and a sample of rows. This helps identify unexpected changes in the source file before cleaning.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place the authorized source CSV at this path or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.dtypes.to_frame("data_type"))

## 3. Remove unused label columns

The project uses `Label_03` as its target. Other label columns and `Stuckzahl` are removed to match the preparation workflow. Missing columns are ignored so the notebook can handle minor differences in source-file versions.

In [ ]:
columns_to_drop = [
    "Label_01", "Label_02", "Label_04", "Label_05",
    "Label_06", "Label_07", "Label_08", "Label_09",
    "Label_10", "Stuckzahl",
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Required target column {TARGET_COLUMN!r} was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Removed columns:", existing_columns_to_drop or "None")
print(f"Remaining columns: {len(df.columns)}")

## 4. Harmonize data types and select usable columns

Boolean values are converted to integers (`False` → `0`, `True` → `1`). The workflow then retains numeric columns and, when present, the time column. Other non-numeric columns are reported and removed.

The target column is checked explicitly because it must remain available for the later classification experiments.

In [ ]:
bool_columns = df.select_dtypes(include=["bool"]).columns.tolist()
if bool_columns:
    df[bool_columns] = df[bool_columns].astype(int)

print(f"Converted {len(bool_columns)} boolean column(s): {bool_columns}")

numeric_mask = df.dtypes.apply(lambda dtype: dtype.kind in "biufc")
time_mask = df.columns.str.lower().eq(TIME_COLUMN.lower())
keep_mask = numeric_mask | time_mask

removed_non_numeric = df.columns[~keep_mask].tolist()
df = df.loc[:, keep_mask].copy()

if TARGET_COLUMN not in df.columns:
    raise TypeError(
        f"{TARGET_COLUMN!r} was removed during type selection. "
        "Check that the target column contains numeric class labels."
    )

print("Removed non-numeric columns:", removed_non_numeric or "None")
print(f"Shape after type selection: {df.shape}")
display(df.head())

## 5. Handle blank and missing values

Whitespace-only strings are treated as missing values. Numeric missing values are then filled with `0`, preserving the original workflow.

**Important:** zero imputation is a modelling assumption, not a universally correct choice. Before using the data in a final model, confirm that a zero is a sensible replacement for missing values in each feature.

In [ ]:
df = df.replace(r"^\s*$", np.nan, regex=True)

missing_before = int(df.isna().sum().sum())
df = df.fillna(0)
missing_after = int(df.isna().sum().sum())

print(f"Missing values before filling: {missing_before:,}")
print(f"Missing values after filling:  {missing_after:,}")
print(f"Final cleaned shape: {df.shape}")

## 6. Export the cleaned dataset

The cleaned dataset retains the original numeric feature scales. It is useful as a transparent intermediate output for subsequent experiments.

In [ ]:
CLEANED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEANED_OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {CLEANED_OUTPUT_PATH.resolve()}")

## 7. Scale numeric features

Min-max scaling maps each feature to the range 0–1. The target `Label_03` and the time column are kept unchanged.

**Evaluation note:** this full-dataset scaled file is intended for exploratory analysis and compatibility with the existing workflow. For an unbiased machine-learning evaluation, fit the scaler only on each training fold (for example, by placing `MinMaxScaler` inside a scikit-learn `Pipeline` used with cross-validation). Do not use this pre-scaled file as a substitute for fold-specific preprocessing when reporting final test metrics.

In [ ]:
feature_columns = [
    col for col in df.columns
    if col not in {TARGET_COLUMN, TIME_COLUMN}
]

df_scaled = df.copy()

if feature_columns:
    scaler = MinMaxScaler()
    df_scaled.loc[:, feature_columns] = scaler.fit_transform(df[feature_columns])
else:
    print("Warning: no feature columns were found to scale.")

SCALED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_scaled.to_csv(SCALED_OUTPUT_PATH, index=False)

print(f"Saved scaled dataset to: {SCALED_OUTPUT_PATH.resolve()}")
display(df_scaled.head())

## 8. Explore feature correlations with the target

This chart shows the 20 lowest and 20 highest Pearson correlations with `Label_03`, excluding the target itself and the time column. Correlation is a descriptive statistic; it does not establish causation or prove that a feature will be useful in a model.

In [ ]:
numeric_data = df_scaled.select_dtypes(include=[np.number]).copy()
numeric_data = numeric_data.drop(columns=[TIME_COLUMN], errors="ignore")

if TARGET_COLUMN not in numeric_data.columns:
    raise KeyError(f"Target column {TARGET_COLUMN!r} is not numeric.")

correlations = (
    numeric_data.corr(numeric_only=True)[TARGET_COLUMN]
    .drop(labels=[TARGET_COLUMN], errors="ignore")
    .dropna()
    .sort_values()
)

if correlations.empty:
    print("No numeric feature correlations could be calculated.")
else:
    extremes = pd.concat([correlations.head(20), correlations.tail(20)]).loc[
        lambda series: ~series.index.duplicated()
    ]

    plt.figure(figsize=(10, max(6, 0.28 * len(extremes))))
    sns.barplot(x=extremes.values, y=extremes.index, orient="h")
    plt.title("Features with the Lowest and Highest Correlation to Label_03")
    plt.xlabel("Pearson correlation coefficient")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    display(extremes.to_frame("correlation_with_Label_03"))

## 9. Explore temperature-feature correlations

This section looks for known temperature-related columns that exist in the dataset and plots their correlation with `Label_03`. If none of the expected columns are present, the notebook reports that rather than failing.

In [ ]:
expected_temperature_columns = [
    "Spindle_motor_temperature",
    "X_Axis_motor_temperature",
    "Z_Axis_Motor_temperature",
    "Y_Axis_Motor_temperature",
    "ToolChange_Motor_temperature",
    "B_Axis_Motor_temperature",
    "C_Axis_Motor_temperature",
    "General_temperature",
]

available_temperature_columns = [
    col for col in expected_temperature_columns if col in df_scaled.columns
]

print(f"Found {len(available_temperature_columns)} expected temperature column(s):")
print(available_temperature_columns or "None")

if available_temperature_columns:
    temperature_correlations = (
        df_scaled[available_temperature_columns + [TARGET_COLUMN]]
        .corr(numeric_only=True)[TARGET_COLUMN]
        .drop(labels=[TARGET_COLUMN], errors="ignore")
        .dropna()
        .sort_values()
    )

    plt.figure(figsize=(10, max(4, 0.45 * len(temperature_correlations))))
    sns.barplot(
        x=temperature_correlations.values,
        y=temperature_correlations.index,
        orient="h",
    )
    plt.title("Temperature-Feature Correlation with Label_03")
    plt.xlabel("Pearson correlation coefficient")
    plt.ylabel("Temperature feature")
    plt.tight_layout()
    plt.show()

    display(temperature_correlations.to_frame("correlation_with_Label_03"))
else:
    print("No expected temperature columns were found; check the source column names.")

## 10. Summary

This notebook creates two local outputs:

- `cleaned_data.csv`: cleaned data before feature scaling.
- `normalized_data.csv`: min-max-scaled features, with `Label_03` and `time` unchanged.

The CSV outputs are generated locally when the notebook is run; they are not automatically included in the GitHub repository. Confirm data permissions before publishing any dataset.

The next modelling stage should use a validation strategy appropriate to the data (including group-wise splitting where needed) and fit preprocessing steps on training data only.